In [0]:
%pip install seaborn

In [0]:
 %restart_python

In [0]:
# ============================================
# Imports, chargement et style
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from matplotlib.patches import FancyBboxPatch
import os, warnings
warnings.filterwarnings("ignore")

# --- Style global "portrait de données" ---
# Palette cinéma premium : or / rouge profond / bleu nuit
PALETTE = ["#C9A227", "#B23A48", "#1B2A41", "#6B8F71", "#DDA15E", "#4A5F8A"]
sns.set_theme(style="whitegrid", palette=PALETTE, context="notebook")
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.titleweight": "bold",
    "axes.titlesize": 15,
    "axes.labelsize": 12,
    "axes.edgecolor": "#888",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
})

# Dossier de sortie pour les images (à committer sur GitHub)
os.makedirs("assets", exist_ok=True)

# --- Chargement ---
df = spark.table("workspace.default.imdb_tmdb_movie_metadata_big_dataset_1_m")
print(f"✅ {df.count():,} films chargés — {len(df.columns)} colonnes")
print("\nColonnes disponibles :")
print(", ".join(df.columns))

#"hero chart" : Top 15 du box-office

In [0]:
from pyspark.sql.functions import col

top15 = (
    df.select("title", "revenue", "release_year")
      .filter(col("revenue").isNotNull() & (col("revenue") > 0))
      .orderBy(col("revenue").desc())
      .limit(15)
      .toPandas()
)
top15["revenue_M"] = top15["revenue"] / 1e6

fig, ax = plt.subplots(figsize=(13, 7))
colors = sns.color_palette("YlOrRd", n_colors=len(top15))[::-1]
bars = ax.barh(top15["title"][::-1], top15["revenue_M"][::-1], color=colors, edgecolor="white")

for bar, val in zip(bars, top15["revenue_M"][::-1]):
    ax.text(val + 15, bar.get_y() + bar.get_height()/2,
            f"${val:,.0f}M", va="center", fontsize=10, fontweight="bold")

ax.set_xlabel("Recettes au box-office (millions USD)")
ax.set_title("🎬 Les 15 plus gros succès du box-office mondial",
             fontsize=17, pad=20, loc="left")
ax.text(0, 1.02, "Source : dataset IMDb × TMDB", transform=ax.transAxes,
        fontsize=10, color="#666", style="italic")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:,.0f}M"))
sns.despine(left=True)
plt.tight_layout()
plt.savefig("assets/01_top_boxoffice.png", dpi=200, bbox_inches="tight")
plt.show()

#Budget vs Recettes 

In [0]:
sample = (
    df.select("title", "budget", "revenue", "release_year")
      .filter((col("budget") > 1e6) & (col("revenue") > 1e6))
      .sample(fraction=0.15, seed=42)   # échantillon pour la lisibilité
      .toPandas()
)
sample["roi"] = sample["revenue"] / sample["budget"]

fig, ax = plt.subplots(figsize=(12, 8))

# Nuage de points avec couleur = ROI
sc = ax.scatter(sample["budget"]/1e6, sample["revenue"]/1e6,
                c=np.log10(sample["roi"]), cmap="RdYlGn",
                s=45, alpha=0.65, edgecolors="white", linewidths=0.4)

# Ligne "break-even" (rentrée = budget)
max_val = max(sample["budget"].max(), sample["revenue"].max()) / 1e6
ax.plot([0, max_val], [0, max_val], "--", color="#888", lw=1.2,
        label="Seuil de rentabilité (Recettes = Budget)")

# Annoter les 3 films au ROI le plus élevé
best = sample.nlargest(3, "roi")
for _, row in best.iterrows():
    ax.annotate(row["title"][:25],
                xy=(row["budget"]/1e6, row["revenue"]/1e6),
                xytext=(row["budget"]/1e6 + 30, row["revenue"]/1e6),
                fontsize=9, fontweight="bold", color="#1B2A41",
                arrowprops=dict(arrowstyle="->", color="#333", lw=0.8))

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Budget (millions USD, échelle log)")
ax.set_ylabel("Recettes (millions USD, échelle log)")
ax.set_title("ROI cinéma : budget vs recettes\n",
             fontsize=17, pad=20, loc="left")
ax.text(0, 1.02, f"Échantillon de {len(sample):,} films — couleur = ROI (vert = rentable)",
        transform=ax.transAxes, fontsize=10, color="#666", style="italic")
cbar = plt.colorbar(sc, ax=ax, shrink=0.8)
cbar.set_label("log10(Recettes / Budget)")
ax.legend(loc="upper left", frameon=True)
plt.tight_layout()
plt.savefig("assets/02_budget_vs_revenue.png", dpi=200, bbox_inches="tight")
plt.show()

#Évolution des genres par décennie

In [0]:
from pyspark.sql.functions import split, explode, year, to_date, regexp_replace

df_decades = (
    df.select("genres_list", "release_date")
      .filter(col("release_date").isNotNull() & col("genres_list").isNotNull())
      .withColumn("annee", year(to_date(regexp_replace("release_date", "/", "-"), "yyyy-MM-dd")))
      .filter((col("annee") >= 1990) & (col("annee") <= 2023))
      .withColumn("decennie", (col("annee") / 10).cast("int") * 10)
      .withColumn("genre", explode(split(regexp_replace("genres_list", "[\\[\\]']", ""), ",\\s*")))
      .groupBy("decennie", "genre").count()
      .toPandas()
)

# Garder les 5 genres les plus fréquents (toutes décennies confondues)
top_genres = df_decades.groupby("genre")["count"].sum().nlargest(5).index
pivot = (df_decades[df_decades["genre"].isin(top_genres)]
         .pivot(index="decennie", columns="genre", values="count").fillna(0))

fig, ax = plt.subplots(figsize=(13, 6))
pivot.plot.area(ax=ax, color=PALETTE[:5], alpha=0.85, linewidth=1.5)
ax.set_xlabel("Décennie")
ax.set_ylabel("Nombre de films produits")
ax.set_title("Évolution de la production par genre (1990–2023)",
             fontsize=17, pad=20, loc="left")
ax.text(0, 1.02, "Top 5 genres les plus représentés dans le dataset",
        transform=ax.transAxes, fontsize=10, color="#666", style="italic")
ax.legend(loc="upper left", frameon=False, ncol=2)
sns.despine()
plt.tight_layout()
plt.savefig("assets/03_genres_decennies.png", dpi=200, bbox_inches="tight")
plt.show()

#Heatmap des corrélations

In [0]:
num_cols = [c for c in ["budget", "revenue", "popularity",
                        "runtime", "vote_average", "vote_count"]
            if c in df.columns]

corr_data = df.select(num_cols).dropna().sample(fraction=0.2, seed=42).toPandas()
corr = corr_data.corr(numeric_only=True)

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(9, 7))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
            annot=True, fmt=".2f", square=True, linewidths=.5,
            cbar_kws={"shrink": .7}, ax=ax,
            annot_kws={"size": 11, "weight": "bold"})
ax.set_title("Corrélations entre variables clés du cinéma",
             fontsize=16, pad=20, loc="left")
ax.text(0, 1.02, f"Échantillon : {len(corr_data):,} films",
        transform=ax.transAxes, fontsize=10, color="#666", style="italic")
plt.tight_layout()
plt.savefig("assets/04_correlation_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

# Dashboard 

In [0]:
from pyspark.sql.functions import avg, count

# Sous-agrégations Spark (rapide) → pandas (petit)
by_year = (df.select("release_date", "revenue", "vote_average")
             .filter(col("release_date").isNotNull())
             .withColumn("annee", year(to_date(regexp_replace("release_date","/","-"),"yyyy-MM-dd")))
             .filter((col("annee") >= 1980) & (col("annee") <= 2023))
             .groupBy("annee")
             .agg(avg("vote_average").alias("note_moy"),
                  count("*").alias("nb_films"))
             .orderBy("annee").toPandas())

top_notes = (df.select("title", "vote_average", "vote_count")
               .filter((col("vote_count") > 5000) & col("vote_average").isNotNull())
               .orderBy(col("vote_average").desc())
               .limit(10).toPandas())

# -- Figure --
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.25)

# (1) Notes moyennes par année
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(by_year["annee"], by_year["note_moy"], color=PALETTE[1], lw=2.5)
ax1.fill_between(by_year["annee"], by_year["note_moy"],
                 alpha=0.2, color=PALETTE[1])
ax1.set_title("Note moyenne IMDb par année", fontsize=13, loc="left")
ax1.set_ylabel("Note moyenne /10")
ax1.set_ylim(4.5, 8)

# (2) Volume de production par année
ax2 = fig.add_subplot(gs[0, 1])
ax2.bar(by_year["annee"], by_year["nb_films"], color=PALETTE[0], edgecolor="white")
ax2.set_title("Volume de films produits par an", fontsize=13, loc="left")
ax2.set_ylabel("Nombre de films")

# (3) Top 10 par note (avec nb de votes)
ax3 = fig.add_subplot(gs[1, 0])
bars = ax3.barh(top_notes["title"].str.slice(0, 30)[::-1],
                top_notes["vote_average"][::-1], color=PALETTE[2])
for b, v in zip(bars, top_notes["vote_average"][::-1]):
    ax3.text(v - 0.15, b.get_y() + b.get_height()/2, f"{v:.1f}",
             va="center", ha="right", color="white", fontweight="bold", fontsize=10)
ax3.set_xlim(0, 10)
ax3.set_title("Top 10 films par note (min. 5k votes)", fontsize=13, loc="left")

# (4) Distribution des recettes (log)
ax4 = fig.add_subplot(gs[1, 1])
rev = (df.select("revenue").filter(col("revenue") > 1e5)
         .sample(fraction=0.1, seed=1).toPandas()["revenue"] / 1e6)
sns.histplot(rev, bins=50, ax=ax4, color=PALETTE[3], kde=True, log_scale=(True, False))
ax4.set_title("Distribution des recettes (log)", fontsize=13, loc="left")
ax4.set_xlabel("Recettes (M USD, échelle log)")
ax4.set_ylabel("Fréquence")

# Titre global
fig.suptitle("Dashboard — Analyse du cinéma mondial (IMDb × TMDB)",
             fontsize=19, fontweight="bold", y=0.98)
fig.text(0.5, 0.955, "4 insights clés extraits du dataset",
         ha="center", fontsize=11, color="#666", style="italic")

plt.savefig("assets/05_dashboard.png", dpi=200, bbox_inches="tight")
plt.show()